# Step 6 - Paper: audit, published protocol reproduced, corrected labels, intervals, ablations

`python -m gids paper` runs these parts in order and rewrites `MyDrive/gotham_ids/reports/paper/PAPER.md` after each one:

| part | | what it does |
|---|---|---|
| `audit` | E0 | measures the labelling problems (copied captures, bots' telnet sessions labelled *TCP Scan*, masscan packets left *Unknown*) and computes the corrected labels |
| `reference` | E1 | reads the CSVs once more and applies the dataset authors' own preprocessing (70 inputs incl. timestamp, TTL, checksums); trains P1-P9 = {their inputs, our features} x {random split as published, our time-blocked split} |
| `score` | E3 | test predictions of the step 3 models, for the confidence intervals |
| `corrected` | E2 | prepare + train + test `xgb_portable` and `xgb_full` again on the corrected labels |
| `ablation` | E4 | header fields only vs earlier-packet context only |
| `baselines` | E5 | logistic regression and the authors' 128-128-64 network on our features |
| `efficiency` | E6 | model size, packets per second |
| `lodo` | E7 | each attacked device type left out of training in turn |
| `seeds` | E3 | the main models with 4 more training seeds |

* **Runtime:** several hours in total, spread over as many Colab sessions as needed. Choose **Runtime -> Change runtime type -> T4 GPU** if available: XGBoost trains on the GPU automatically (CPU also works, slower).
* **Disconnected?** Reconnect and run cells 1, 2 and 4 again. Every finished part, model, prediction and table is kept on Drive and skipped; the CSV pass of `reference` continues inside the file where it stopped.
* **Only some parts:** set `PARTS` in cell 1 (e.g. `reference,score`); empty = all.
* **Results:** copy what cell 6 prints into the chat.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}
PARTS = ""  # e.g. "reference,score"; empty = every part, in order  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 2 min)
!python -m gids selftest

In [ ]:
#@title 4. Run the paper parts (resumable: after a disconnect run cells 1, 2 and 4 again)
ARGS = f"--parts {PARTS.strip()}" if PARTS.strip() else ""
!python -u -m gids paper $ARGS

In [ ]:
#@title 5. Show PAPER.md and its figures
import glob
from IPython.display import Image, Markdown, display
P = f"{WORK_DIR}/reports/paper"
display(Markdown(open(f"{P}/PAPER.md").read()))  # Colab cannot show the document's local images inline ...
for f in sorted(glob.glob(f"{P}/figures/*.png")):  # ... so they are shown here
    print(f.rsplit("/", 1)[-1])
    display(Image(f, width=760))

In [ ]:
#@title 6. Plain text of PAPER.md and the audit (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/paper/PAPER.md").read())
import os
if os.path.exists(f"{WORK_DIR}/reports/audit/AUDIT.md"):
    print(open(f"{WORK_DIR}/reports/audit/AUDIT.md").read())

In [ ]:
#@title 7. Download the paper folder (document, figures, tables) as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/gotham_paper", "zip", f"{WORK_DIR}/reports/paper")
files.download("/content/gotham_paper.zip")

In [ ]:
#@title 8. Optional: check the re-implementation of the authors' preprocessing on real packets (a few minutes)
# Runs the authors' notebook functions and gids/reference.py on the first rows of one real CSV and lists every
# model input that differs ("none" expected). This clones their repository and executes its notebook code.
import os, subprocess
if not os.path.isdir("/content/labeller"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/othmbela/gotham-network-packet-labeller",
                    "/content/labeller"], check=True)
!python tools/check_reference.py --zip "$ZIP_PATH" --labeller /content/labeller --rows 200000

### If something goes wrong
* `status` shows which paper parts are done; `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* A failed part is logged and the next part still runs; run cell 4 again to retry it.
* Per-part outputs: `reports/audit/` (E0), `reports/paper/reference/` (E1), `reports_corrected/` (E2 reports), `reports/paper/*.csv`, test predictions `models/<experiment>-<hash>/test_pred.npy`.

In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30